# SafeCityAI: Custom YOLOv5 Object Detection Pipeline

This notebook guides you through training a custom object detector using **YOLOv5** to identify traffic safety compliance for **SafeCityAI**.

### Classes of Interest
- **Helmet** (Class 0): Motorcyclist/cyclist wearing a helmet.
- **No_Helmet** (Class 1): Motorcyclist/cyclist NOT wearing a helmet (traffic violation).
- **License_Plate** (Class 2): Vehicle license plate for automated ticket generation.
- **Seatbelt** (Class 3): Driver/passenger wearing a seatbelt.
- **No_Seatbelt** (Class 4): Driver/passenger NOT wearing a seatbelt (traffic violation).

### Workflow Steps
1. **Environment Setup**: GPU check, cloning YOLOv5 repository, and installing requirements.
2. **Dataset Configuration**: Loading the annotated dataset (Roboflow or custom zip upload) and setting up `data.yaml`.
3. **Model Configuration & Hyperparameters**: Modifying augmentation settings (mosaic, flip, etc.) and anchor values.
4. **Training**: Launching transfer learning using YOLOv5 small (`yolov5s.pt`) weights.
5. **Evaluation**: Monitoring metrics like loss and Precision-Recall, and evaluating the Mean Average Precision (mAP@0.5).
6. **Inference & Visualization**: Executing inference on test street images and videos.
7. **Exporting Weights**: Downloading `best.pt` for Flask API deployment.

## Step 1: Environment Setup

First, we verify that a GPU is allocated to our Colab session for faster training. Then we clone the official YOLOv5 repository and install its dependencies.

In [1]:
# Verify GPU allocation
!nvidia-smi

'nvidia-smi' n'est pas reconnu en tant que commande interne
ou externe, un programme excutable ou un fichier de commandes.


In [ ]:
# Clone YOLOv5 repository
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5

# Install dependencies
!pip install -r requirements.txt

import torch
print(f"Setup complete. Using PyTorch {torch.__version__} with device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## Step 2: Download & Structure the Dataset

You can import your annotated dataset from **Roboflow** directly using their Python SDK. 
Alternatively, you can upload a zip file of your dataset containing `images/` and `labels/` folders divided into `train` and `val` subdirectories.

In [ ]:
# Install Roboflow SDK and download dataset
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="WoRazYHMndMcsbTA87UW")
project = rf.workspace("ratovoarisoaricardo-gmail-com").project("rider-helmet-no-helmet-lic-plate")
dataset = project.version(1).download("yolov5")

### Verify `data.yaml` Configuration

We need a `data.yaml` that tells YOLOv5 where to find training and validation images, the number of classes (`nc`), and their names.

In [ ]:
# Roboflow automatically generates the data.yaml file in the dataset directory.
print(f"Data YAML location: {dataset.location}/data.yaml")

with open(f"{dataset.location}/data.yaml", 'r') as f:
    print(f.read())

## Step 3: Model Training

We start training with the following arguments:
- `--img`: Input image size (typically `640` pixels).
- `--batch`: Batch size (e.g. `16` or `32` depending on GPU memory).
- `--epochs`: Number of training iterations (epochs). We recommend starting with `50` to `100` epochs for fine-tuning.
- `--data`: Path to `data.yaml` configuration file.
- `--weights`: Path to initial weights. We start with pre-trained COCO weights `yolov5s.pt` to leverage transfer learning.
- `--cache`: Cache images for faster training.

In [ ]:
# Train YOLOv5s on custom data
!python train.py --img 640 --batch 16 --epochs 50 --data {dataset.location}/data.yaml --weights yolov5s.pt --cache

## Step 4: Monitoring and Evaluation

YOLOv5 automatically logs stats to TensorBoard. We can launch TensorBoard to monitor training losses (box loss, objectness loss, class loss) and metrics like Precision, Recall, and Mean Average Precision (`mAP@0.5`).

In [ ]:
# Start Tensorboard inside the Notebook
%load_ext tensorboard
%tensorboard --logdir runs/train

### Evaluate on Validation Set

Run validation to check the final mAP score on the validation split.

In [ ]:
# Validate using the trained best.pt weights
!python val.py --weights runs/train/exp/weights/best.pt --data {dataset.location}/data.yaml --img 640

## Step 5: Test Inference on Images & Videos

Now we run the `detect.py` script to perform inference on unseen images or videos. The results will be saved in the directory `runs/detect/`.

In [ ]:
# Run inference on a test image
!python detect.py --weights runs/train/exp/weights/best.pt --img 640 --conf 0.4 --source {dataset.location}/valid/images/

# Run inference on a traffic video (outputs a video with bounding boxes drawn)
# !python detect.py --weights runs/train/exp/weights/best.pt --source path_to_video.mp4 --view-img

In [ ]:
# Visualizing the predictions
import cv2
import matplotlib.pyplot as plt
import glob
import os
%matplotlib inline

# Find the latest detection folder
detect_dirs = glob.glob('runs/detect/exp*')
if detect_dirs:
    latest_dir = max(detect_dirs, key=os.path.getctime)
    detected_images = glob.glob(os.path.join(latest_dir, '*.jpg'))
    
    if detected_images:
        img_path = detected_images[0]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(10, 10))
        plt.imshow(img)
        plt.axis('off')
        plt.title("Inference Result")
        plt.show()
    else:
        print("No detected images found.")
else:
    print("No detection directories found.")

## Step 6: Export Trained Weights for API Deployment

We can download our custom `best.pt` file from Colab's file tree, or run the cell below to download it directly. This file is required by our API server (`server.py`).

In [ ]:
# Code to download best.pt locally
try:
    from google.colab import files
    files.download('runs/train/exp/weights/best.pt')
    print("Triggered download for best.pt")
except ImportError:
    print("Google Colab files utility not available. Please download best.pt from: runs/train/exp/weights/best.pt")